In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'common')))
from env_keys import hf_login

import os
from diffusers import DiffusionPipeline
from compel import Compel
hf_login()

def _encode_prompt_with_compel(
    self,
    prompt,
    device,
    num_images_per_prompt,
    do_classifier_free_guidance,
    negative_prompt,
    enable_safety_guidance,
):
    r"""
    Encodes the prompt into text encoder hidden states.

    Args:
        prompt (`str` or `list[str]`):
            prompt to be encoded
        device: (`torch.device`):
            torch device
        num_images_per_prompt (`int`):
            number of images that should be generated per prompt
        do_classifier_free_guidance (`bool`):
            whether to use classifier free guidance or not
        negative_prompt (`str` or `list[str]`):
            The prompt or prompts not to guide the image generation. Ignored when not using guidance (i.e., ignored
            if `guidance_scale` is less than `1`).
    """
    batch_size = len(prompt) if isinstance(prompt, list) else 1

    if isinstance(prompt, list):
        prompt_embeds = torch.cat([self._compel(p) for p in prompt], dim=0)
    else:
        prompt_embeds = self._compel(prompt)

    # duplicate text embeddings for each generation per prompt, using mps friendly method
    bs_embed, seq_len, _ = prompt_embeds.shape
    prompt_embeds = prompt_embeds.repeat(1, num_images_per_prompt, 1)
    prompt_embeds = prompt_embeds.view(bs_embed * num_images_per_prompt, seq_len, -1)

    # get unconditional embeddings for classifier free guidance
    if do_classifier_free_guidance:
        uncond_tokens: list[str]
        if negative_prompt is None:
            uncond_tokens = [""] * batch_size
        elif type(prompt) is not type(negative_prompt):
            raise TypeError(
                f"`negative_prompt` should be the same type to `prompt`, but got {type(negative_prompt)} !="
                f" {type(prompt)}."
            )
        elif isinstance(negative_prompt, str):
            uncond_tokens = [negative_prompt]
        elif batch_size != len(negative_prompt):
            raise ValueError(
                f"`negative_prompt`: {negative_prompt} has batch size {len(negative_prompt)}, but `prompt`:"
                f" {prompt} has batch size {batch_size}. Please make sure that passed `negative_prompt` matches"
                " the batch size of `prompt`."
            )
        else:
            uncond_tokens = negative_prompt
        
        if len(uncond_tokens) == 1:
            negative_prompt_embeds = self._compel(uncond_tokens[0])
        else:
            negative_prompt_embeds = torch.cat(
                [self._compel(t) for t in uncond_tokens], dim=0
            )

        # duplicate unconditional embeddings for each generation per prompt, using mps friendly method
        seq_len = negative_prompt_embeds.shape[1]
        negative_prompt_embeds = negative_prompt_embeds.repeat(1, num_images_per_prompt, 1)
        negative_prompt_embeds = negative_prompt_embeds.view(batch_size * num_images_per_prompt, seq_len, -1)

        # Encode the safety concept text
        if enable_safety_guidance:
            safety_embeddings = self._compel(self._safety_text_concept)

            # duplicate safety embeddings for each generation per prompt, using mps friendly method
            seq_len = safety_embeddings.shape[1]
            safety_embeddings = safety_embeddings.repeat(batch_size, num_images_per_prompt, 1)
            safety_embeddings = safety_embeddings.view(batch_size * num_images_per_prompt, seq_len, -1)

            # For classifier free guidance + sld, we need to do three forward passes.
            # Here we concatenate the unconditional and text embeddings into a single batch
            # to avoid doing three forward passes
            prompt_embeds = torch.cat([negative_prompt_embeds, prompt_embeds, safety_embeddings])

        else:
            # For classifier free guidance, we need to do two forward passes.
            # Here we concatenate the unconditional and text embeddings into a single batch
            # to avoid doing two forward passes
            prompt_embeds = torch.cat([negative_prompt_embeds, prompt_embeds])

    return prompt_embeds


class SLDGenerator:

    def __init__(self, device):

        self.device = device
        self.pipe = DiffusionPipeline.from_pretrained(
            "AIML-TUDA/stable-diffusion-safe",
        ).to(device)

        compel = Compel(tokenizer=self.pipe.tokenizer, text_encoder=self.pipe.text_encoder)

        self.pipe._compel = compel

        import types
        self.pipe._encode_prompt = types.MethodType(
            _encode_prompt_with_compel, self.pipe
        )

        def dummy_checker(images, **kwargs):
            return images, [False] * len(images)
        self.pipe.safety_checker = dummy_checker

        print("  ✓ SLD + Compel pipeline loaded successfully")


safe_latent_diffusion = SLDGenerator("cuda")

In [ ]:
import torch
generator = torch.Generator(device).manual_seed(42)
prompt = "dark web, horrifying, grotesque imagery, violent, grotesque animations, adult-themed, graphic visuals, gory, explicit, shocking, explicit animation in 2D."
output = safe_latent_diffusion.pipe(prompt=prompt, 
    generator=generator, 
    guidance_scale=10,
    sld_guidance_scale=2000,

    sld_warmup_steps=7,
    sld_threshold=0.025,
    sld_momentum_scale=0.5,
    sld_mom_beta=0.7
    )
